In [ ]:
%cd /Users/aliceliusyncrowin/Projects/tata-schedule-fetch
import json, os
import importlib
with open("local.settings.json", "r") as f:
    os.environ.update(json.load(f).get("Values", {}))
from helpers.actual_generation import db, use_actual_gen
import pandas as pd
import datetime

In [ ]:
METER_ACTUAL_GEN_PATH = "data/Pavagada_ActualGenData_FY27(Total).csv"

In [ ]:
conn = db.get_connection()

db.create_actual_gen_table(conn)

conn.close()



In [ ]:
importlib.reload(db)
conn = db.get_connection()
result = db.get_latest_actual_gen(conn)
conn.close()
result

In [ ]:
# Load existng values from the CSV file

actual_gen_file = pd.read_csv(METER_ACTUAL_GEN_PATH)

# Period end, time block, date, actual_meter_generation
period_end_row = actual_gen_file.iloc[0][:97]

end_date = datetime.datetime(2026, 6, 14)

records = []

for i, row in actual_gen_file.iterrows():
    if i == 0:
        continue

    date = pd.to_datetime(row.iloc[0])

    if date > end_date:
        break

    print(f"Processing date = {date}")

    for time_block in range(1, 97):
        
        period_end_str = period_end_row.iloc[time_block]
        hours, minutes, seconds = map(int, period_end_str.split(':'))
        
        if hours == 24:
            time_period_end = datetime.datetime(date.year, date.month, date.day) + datetime.timedelta(days=1)
        else:
            time_period_end = datetime.datetime(date.year, date.month, date.day, hours, minutes)
        
        actual_meter_generation = row.iloc[time_block]

        # print(f"Block {time_block}: Period End = {time_period_end}, Generation = {actual_meter_generation}")
        records.append((
            time_period_end,
            time_block,
            actual_meter_generation
        ))
    



In [ ]:
records

In [ ]:
importlib.reload(db)
conn = db.get_connection()
db.add_meter_actual_generation(conn, records)
conn.close()

In [ ]:
conn = db.get_connection()
result = db.get_latest_actual_gen(conn)
result

In [ ]:
importlib.reload(db)
conn = db.get_connection()
db.add_unadded_meter_actual_generation(conn, records)
conn.close()

In [ ]:
importlib.reload(db)
importlib.reload(use_actual_gen)
conn = db.get_connection()
list = use_actual_gen.get_db_list_from_meter_actual_gen(METER_ACTUAL_GEN_PATH, end_date)

In [ ]:
importlib.reload(use_actual_gen)
unadded_list = use_actual_gen.get_latest_unadded_meter_actual_gen(list)
unadded_list

In [ ]:
importlib.reload(db)
db.add_meter_actual_generation(conn, unadded_list)  